# Build Silver Maintenance Events

This notebook promotes JetOps maintenance events from Bronze into a cleaner Silver Delta dataset.

Execution behavior:
- Databricks: reads Bronze Delta, applies cleaning and deduplication rules, and writes Silver Delta.
- Local VS Code notebook: reconstructs Bronze-like rows from the latest raw capture file and previews the Silver transformation without writing Delta.

Current Silver rules:
- keep only records with required identifiers and timestamps
- standardize key text fields
- deduplicate on `event_id`, keeping the latest event
- derive analytics-friendly timestamp and date columns

In [ ]:
import os

storage_account = os.getenv("JETOPS_STORAGE_ACCOUNT", "stherbalifedev001")
raw_container = os.getenv("JETOPS_RAW_CONTAINER", "raw")
bronze_container = os.getenv("JETOPS_BRONZE_CONTAINER", "bronze")
silver_container = os.getenv("JETOPS_SILVER_CONTAINER", "silver")
secret_scope = os.getenv("JETOPS_SECRET_SCOPE", "herbalife-storage")
raw_secret_key = os.getenv("JETOPS_SECRET_KEY", "raw-sas-token")
bronze_secret_key = os.getenv("JETOPS_BRONZE_SECRET_KEY", raw_secret_key)
silver_secret_key = os.getenv("JETOPS_SILVER_SECRET_KEY", raw_secret_key)
storage_account_key_secret = os.getenv("JETOPS_STORAGE_ACCOUNT_KEY_SECRET", "storage-account-key")
storage_auth_mode = os.getenv("JETOPS_STORAGE_AUTH_MODE", "account_key")
eventhub_namespace = os.getenv("JETOPS_EVENTHUB_NAMESPACE", "evh-herbalife-dev")
eventhub_name = os.getenv("JETOPS_EVENTHUB_NAME", "jetops-maintenance-events-dev")
capture_root = os.getenv("JETOPS_CAPTURE_ROOT", "jetops-maintenance")
resource_group = os.getenv("JETOPS_RESOURCE_GROUP", "rg-herbalife-dev-core")
bronze_dataset = os.getenv("JETOPS_BRONZE_DATASET", "jetops/maintenance_events")
silver_dataset = os.getenv("JETOPS_SILVER_DATASET", "jetops/maintenance_events")
write_mode = os.getenv("JETOPS_SILVER_WRITE_MODE", "overwrite")
az_cli = os.getenv("AZURE_CLI_PATH", r"C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin\az.cmd")

bronze_delta_path = f"wasbs://{bronze_container}@{storage_account}.blob.core.windows.net/{bronze_dataset}"
silver_delta_path = f"wasbs://{silver_container}@{storage_account}.blob.core.windows.net/{silver_dataset}"
raw_capture_prefix = f"{capture_root}/{eventhub_namespace}/{eventhub_name}"

is_databricks = "dbutils" in globals() and "spark" in globals()
print(f"Execution mode: {'databricks' if is_databricks else 'local'}")
print(f"Bronze Delta path: {bronze_delta_path}")
print(f"Silver Delta path: {silver_delta_path}")
print(f"Write mode: {write_mode}")
print(f"Databricks storage auth mode: {storage_auth_mode}")

if is_databricks:
    if storage_auth_mode == "sas":
        bronze_sas_token = dbutils.secrets.get(scope=secret_scope, key=bronze_secret_key)
        silver_sas_token = dbutils.secrets.get(scope=secret_scope, key=silver_secret_key)
        spark.conf.set(
            f"fs.azure.sas.{bronze_container}.{storage_account}.blob.core.windows.net",
            bronze_sas_token,
        )
        spark.conf.set(
            f"fs.azure.sas.{silver_container}.{storage_account}.blob.core.windows.net",
            silver_sas_token,
        )
    else:
        storage_account_key = dbutils.secrets.get(scope=secret_scope, key=storage_account_key_secret)
        spark.conf.set(
            f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
            storage_account_key,
        )
else:
    print("Local mode will preview Silver rows from the latest captured Avro file.")

Execution mode: local
Bronze Delta path: wasbs://bronze@stherbalifedev001.blob.core.windows.net/jetops/maintenance_events
Silver Delta path: wasbs://silver@stherbalifedev001.blob.core.windows.net/jetops/maintenance_events
Write mode: overwrite
Local mode will preview Silver rows from the latest captured Avro file.


In [ ]:
import json
import subprocess
import tempfile
from datetime import UTC, datetime
from pathlib import Path

if is_databricks:
    from pyspark.sql.functions import col, current_timestamp, lower, row_number, to_date, to_timestamp, trim, upper, initcap
    from pyspark.sql.window import Window

    bronze_df = spark.read.format("delta").load(bronze_delta_path)
    silver_base_df = (
        bronze_df
        .withColumn("event_timestamp", to_timestamp(col("event_timestamp")))
        .withColumn("enqueued_time_utc", to_timestamp(col("enqueued_time_utc")))
        .withColumn("bronze_loaded_at", to_timestamp(col("bronze_loaded_at")))
        .withColumn("inspection_date", to_date(col("inspection_date")))
        .withColumn("event_id", trim(col("event_id")))
        .withColumn("tail_number", upper(trim(col("tail_number"))))
        .withColumn("airport_code", upper(trim(col("airport_code"))))
        .withColumn("hangar", upper(trim(col("hangar"))))
        .withColumn("component", initcap(trim(col("component"))))
        .withColumn("maintenance_type", initcap(trim(col("maintenance_type"))))
        .withColumn("status", initcap(trim(col("status"))))
        .withColumn("severity", initcap(trim(col("severity"))))
        .withColumn("schema_version", trim(col("schema_version")))
        .withColumn("ingestion_source", lower(trim(col("ingestion_source"))))
        .withColumn("event_date", to_date(col("event_timestamp")))
        .filter(col("event_id").isNotNull())
        .filter(col("event_timestamp").isNotNull())
        .filter(col("tail_number").isNotNull())
        .filter(col("component").isNotNull())
    )

    silver_window = Window.partitionBy("event_id").orderBy(
        col("event_timestamp").desc_nulls_last(),
        col("bronze_loaded_at").desc_nulls_last(),
        col("sequence_number").desc_nulls_last()
    )

    silver_df = (
        silver_base_df
        .withColumn("silver_row_rank", row_number().over(silver_window))
        .filter(col("silver_row_rank") == 1)
        .drop("silver_row_rank")
        .withColumn("silver_loaded_at", current_timestamp())
    )

    writer = (
        silver_df.write
        .format("delta")
        .mode(write_mode)
        .option("overwriteSchema", "true")
    )
    if write_mode != "overwrite":
        writer = writer.option("mergeSchema", "true")

    (
        writer
        .partitionBy("event_date")
        .save(silver_delta_path)
    )

    print(f"Wrote Silver Delta dataset to {silver_delta_path}")
    print(f"Rows written: {silver_df.count()}")
else:
    try:
        from fastavro import reader
    except ImportError as exc:
        raise ImportError(
            "Local mode requires fastavro in the notebook kernel. Install it before running this cell."
        ) from exc

    account_key = subprocess.check_output(
        [
            az_cli,
            "storage",
            "account",
            "keys",
            "list",
            "--resource-group",
            resource_group,
            "--account-name",
            storage_account,
            "--query",
            "[0].value",
            "-o",
            "tsv",
        ],
        text=True,
    ).strip()

    file_list = subprocess.check_output(
        [
            az_cli,
            "storage",
            "fs",
            "file",
            "list",
            "--account-name",
            storage_account,
            "--account-key",
            account_key,
            "--file-system",
            raw_container,
            "--path",
            raw_capture_prefix,
            "--exclude-dir",
            "-o",
            "json",
        ],
        text=True,
    )
    files = json.loads(file_list)
    avro_files = sorted(file_info["name"] for file_info in files if file_info["name"].endswith(".avro"))
    if not avro_files:
        raise FileNotFoundError(f"No Avro capture files found under {raw_capture_prefix}")

    latest_file = avro_files[-1]
    with tempfile.TemporaryDirectory() as temp_dir:
        local_file = Path(temp_dir) / Path(latest_file).name
        subprocess.run(
            [
                az_cli,
                "storage",
                "fs",
                "file",
                "download",
                "--account-name",
                storage_account,
                "--account-key",
                account_key,
                "--file-system",
                raw_container,
                "--path",
                latest_file,
                "--destination",
                str(local_file),
                "--overwrite",
                "true",
            ],
            check=True,
            capture_output=True,
            text=True,
        )

        with local_file.open("rb") as handle:
            records = list(reader(handle))

    def _parse_event_timestamp(value):
        if not value:
            return None
        return datetime.fromisoformat(value.replace("Z", "+00:00"))

    latest_by_event_id = {}
    for record in records:
        body_json = record["Body"].decode("utf-8")
        payload = json.loads(body_json)
        event_id = (payload.get("event_id") or "").strip()
        event_timestamp = _parse_event_timestamp(payload.get("event_time_utc"))
        tail_number = (payload.get("tail_number") or "").strip().upper()
        component = (payload.get("component") or "").strip().title()
        if not event_id or not event_timestamp or not tail_number or not component:
            continue

        silver_row = {
            "sequence_number": record.get("SequenceNumber"),
            "offset": record.get("Offset"),
            "enqueued_time_utc": record.get("EnqueuedTimeUtc"),
            "system_properties_json": json.dumps(record.get("SystemProperties", {})),
            "properties_json": json.dumps(record.get("Properties", {})),
            "body_json": body_json,
            **payload,
            "event_id": event_id,
            "tail_number": tail_number,
            "airport_code": (payload.get("airport_code") or "").strip().upper(),
            "hangar": (payload.get("hangar") or "").strip().upper(),
            "component": component,
            "maintenance_type": (payload.get("maintenance_type") or "").strip().title(),
            "status": (payload.get("status") or "").strip().title(),
            "severity": (payload.get("severity") or "").strip().title(),
            "schema_version": (payload.get("schema_version") or "").strip(),
            "ingestion_source": (payload.get("ingestion_source") or "").strip().lower(),
            "source_file": latest_file,
            "bronze_loaded_at": datetime.now(UTC).isoformat().replace("+00:00", "Z"),
            "event_timestamp": event_timestamp.isoformat().replace("+00:00", "Z"),
            "event_date": event_timestamp.date().isoformat(),
            "silver_loaded_at": datetime.now(UTC).isoformat().replace("+00:00", "Z"),
        }

        current = latest_by_event_id.get(event_id)
        if current is None or silver_row["event_timestamp"] > current["event_timestamp"] or (
            silver_row["event_timestamp"] == current["event_timestamp"] and (silver_row["sequence_number"] or -1) > (current["sequence_number"] or -1)
        ):
            latest_by_event_id[event_id] = silver_row

    silver_preview = list(latest_by_event_id.values())[:10]
    print(f"Latest raw file: {latest_file}")
    print(f"Previewing {len(silver_preview)} cleaned Silver rows after deduplication.")
    silver_preview

Latest raw file: jetops-maintenance/evh-herbalife-dev/jetops-maintenance-events-dev/1/2026/04/05/03/49/50.avro
Previewing 10 cleaned Silver rows after deduplication.


In [3]:
if is_databricks:
    from pyspark.sql.functions import col

    silver_delta_df = spark.read.format("delta").load(silver_delta_path)
    print(f"Silver row count: {silver_delta_df.count()}")
    display(silver_delta_df.orderBy(col("event_timestamp").desc()))
else:
    print("Local mode does not write Delta. Use the Silver preview from Cell 3 to validate the cleaned output shape.")

Local mode does not write Delta. Use the Silver preview from Cell 3 to validate the cleaned output shape.
